# Explanatory Charts — TMDB Fantasy Movie Dataset

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import duckdb
from scipy.ndimage import uniform_filter1d
from src.core.config import DUCKDB_PATH

OUT_DIR = "reports/figures/explanatory"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Shared style ─────────
plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.size":         11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    15,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "figure.facecolor":  "#FFFFFF",
    "axes.facecolor":    "#FFFFFF",
    "axes.grid":         True,
    "grid.color":        "#E8E8E8",
    "grid.linewidth":    0.6,
    "text.color":        "#1A1A1A",
    "axes.labelcolor":   "#1A1A1A",
    "xtick.color":       "#1A1A1A",
    "ytick.color":       "#1A1A1A",
})

BLUE   = "#2C6FAD"
ORANGE = "#E07B39"
GREEN  = "#2E8B57"
RED    = "#C0392B"
PURPLE = "#7B4FA6"
TEAL   = "#1A7F74"
DARK   = "#1A1A1A"
GREY   = "#888888"

ANNOTATION_BOX = dict(
    boxstyle="round,pad=0.4",
    facecolor="#FFFFFF",
    edgecolor="#CCCCCC",
    alpha=0.95
)

# ── Load data ────────
conn = duckdb.connect(DUCKDB_PATH, read_only=False)
conn.execute("CHECKPOINT")

movies  = conn.execute("SELECT * FROM movies").fetchdf()
details = conn.execute("SELECT * FROM movie_details").fetchdf()
genres  = conn.execute("SELECT * FROM genres").fetchdf()
mg      = conn.execute("SELECT * FROM movie_genres").fetchdf()
conn.close()

df = movies.merge(details, on="movie_id", how="left", suffixes=("", "_d"))
for col in ["budget", "revenue", "runtime"]:
    df[col] = df[col].replace(0, np.nan)

df = df[df["release_year"].between(1980, 2024)]
mg_named = mg.merge(genres, on="genre_id")


# chart 1
# Fantasy film ratings were highest before the franchise era.
# As production volume exploded post-2010 as with all new things it first soared, just to crash as the average quality declined.

In [2]:
yearly = (
    df.groupby("release_year")
    .agg(avg_rating=("rating", "mean"), count=("movie_id", "count"))
    .reset_index()
)
yearly = yearly[yearly["count"] >= 3]
yearly["smoothed"] = uniform_filter1d(yearly["avg_rating"], size=5)

fig, ax = plt.subplots(figsize=(14, 7))
fig.subplots_adjust(bottom=0.18)


era_bands = [
    (1980, 1995, "#EBF2FA", "Pre-CGI Era\n1980–1994"),
    (1995, 2010, "#EAF4EE", "CGI Expansion\n1995–2009"),
    (2010, 2024, "#FDF3EC", "Franchise Era\n2010–2024"),
]
for x0, x1, color, label in era_bands:
    ax.axvspan(x0, x1, color=color, alpha=1.0, zorder=0)


ax.autoscale()
y_min = ax.get_ylim()[0]
for x0, x1, color, label in era_bands:
    ax.text((x0 + x1) / 2, y_min + 0.05, label,
            ha="center", va="bottom", fontsize=8.5,
            color=GREY, style="italic", zorder=4)


ax.scatter(
    yearly["release_year"], yearly["avg_rating"],
    s=yearly["count"] * 3, color=BLUE, alpha=0.30,
    zorder=2, label="Annual average  (bubble size = no. of movies)"
)


ax.plot(
    yearly["release_year"], yearly["smoothed"],
    color=DARK, linewidth=2.8, zorder=3, label="5-year smoothed trend"
)


peak = yearly.loc[yearly["smoothed"].idxmax()]
ax.scatter(peak["release_year"], peak["smoothed"],
           s=200, color=GREEN, zorder=5, marker="*")
ax.annotate(
    f"Highest rated period\nAvg {peak['smoothed']:.2f} / 10",
    xy=(peak["release_year"], peak["smoothed"]),
    xytext=(peak["release_year"] - 1, peak["smoothed"] + 0.15),
    fontsize=9.5, color=GREEN, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.5),
    bbox={**ANNOTATION_BOX, "edgecolor": GREEN},
    zorder=6
)

trough = yearly.loc[yearly["smoothed"].idxmin()]
ax.scatter(trough["release_year"], trough["smoothed"],
           s=150, color=RED, zorder=5, marker="v")
ax.annotate(
    f"Lowest rated period\nAvg {trough['smoothed']:.2f} / 10",
    xy=(trough["release_year"], trough["smoothed"]),
    xytext=(trough["release_year"] + 1, trough["smoothed"] - 0.1),
    fontsize=9.5, color=RED, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=RED, lw=1.5),
    bbox={**ANNOTATION_BOX, "edgecolor": RED},
    zorder=6
)

post = yearly[yearly["release_year"] >= 2010]
pre  = yearly[yearly["release_year"] <  2010]
ax.annotate(
    f"{post['count'].sum()} movies released 2010–2024\nvs {pre['count'].sum()} in 1980–2009\n"
    "More volume — lower average quality",
    xy=(2018, post["smoothed"].mean()),
    xytext=(2009, post["smoothed"].mean() + 0.25),
    fontsize=9, color=ORANGE, fontweight="bold", style="italic",
    arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1.3),
    bbox={**ANNOTATION_BOX, "edgecolor": ORANGE},
    zorder=6
)

ax.set_xlim(1979, 2025)
ax.set_xlabel("Release year", labelpad=8)
ax.set_ylabel("Average audience rating (out of 10)", labelpad=8)
ax.set_title("More Fantasy Films, Lower Scores", pad=14, loc="left", fontsize=16, y=1.04)
ax.text(0, 1.01,
        "As production volume surged after 2010 with the new CGI tools (Transformers ERA), \n average audience ratings first soared, and later declined — "
        "suggesting quality became harder to maintain at scale.",
        transform=ax.transAxes, fontsize=10, color=GREY, style="italic")
ax.legend(loc="upper right", framealpha=0.95, fontsize=9, edgecolor="#CCCCCC")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))

fig.text(
    0.5, 0.065,
    "Takeaway: The appearance of new age CGI tools shows a clear impact 2010 with movies like transformers being the spearhead. "+"\n"+
    "While later with the franchise era in 2020ish brought volume — but average ratings dropped noticeably, as the cool factor went away.",
    ha="center", va="bottom", fontsize=9.5, color=DARK, style="italic",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#F5F5F5", edgecolor="#CCCCCC")
)

out1 = f"{OUT_DIR}/charts1_rating_over_time.png"
fig.tight_layout()
fig.savefig(out1, dpi=180)
plt.close()



C:\Users\strom\AppData\Local\Temp\ipykernel_24996\1298779354.py:102: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


# 2
#  Action and Adventure have increasingly crowded the fantasy genre space for most of its lifetime
# Niche genres like Horror and Animation have shrunk in share over the decades

In [3]:
df["decade"] = (df["release_year"] // 10 * 10).astype("Int64")

genre_decade = (
    mg_named
    .merge(df[["movie_id", "decade"]], on="movie_id")
    .dropna(subset=["decade"])
    .groupby(["decade", "genre_name"])
    .agg(movie_count=("movie_id", "nunique"))
    .reset_index()
)
totals = genre_decade.groupby("decade")["movie_count"].transform("sum")
genre_decade["share"] = genre_decade["movie_count"] / totals * 100

decades = sorted(genre_decade["decade"].unique())

pivot = genre_decade.pivot_table(
    index="genre_name", columns="decade", values="share", fill_value=0
)
top_genres = pivot.std(axis=1).nlargest(6).index.tolist()
COLORS = [BLUE, ORANGE, GREEN, RED, PURPLE, TEAL]

fig, ax = plt.subplots(figsize=(14, 7))
fig.subplots_adjust(right=0.76, bottom=0.18)

for genre, color in zip(top_genres, COLORS):
    sub = (
        genre_decade[genre_decade["genre_name"] == genre]
        .sort_values("decade")
        .reset_index(drop=True)
    )

    ax.plot(
        sub["decade"], sub["share"],
        marker="o", linewidth=2.5, color=color, zorder=3,
        markersize=7, markerfacecolor="white",
        markeredgewidth=2.2, markeredgecolor=color
    )

    last = sub.iloc[-1]
    ax.text(
        last["decade"] + 1.8, last["share"],
        f"{genre}  {last['share']:.1f}%",
        fontsize=9, color=color, va="center", fontweight="bold"
    )

    if len(sub) >= 2:
        sub["diff"] = sub["share"].diff()
        abs_max = sub["diff"].abs().max()
        if abs_max > 4:
            max_idx = sub["diff"].abs().idxmax()
            row  = sub.loc[max_idx]
            prev = sub.loc[max_idx - 1]
            sign = "+" if row["diff"] > 0 else ""
            ax.annotate(
                f"{sign}{row['diff']:.1f}%point shift\n"
                f"{int(prev['decade'])}s → {int(row['decade'])}s",
                xy=(row["decade"], row["share"]),
                xytext=(row["decade"] - 7, row["share"] + 4),
                fontsize=8.5, color=color, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=color, lw=1.3),
                bbox={**ANNOTATION_BOX, "edgecolor": color},
                zorder=6
            )

ax.set_xticks(decades)
ax.set_xticklabels([f"{int(d)}s" for d in decades], fontsize=10)
ax.set_xlim(decades[0] - 3, decades[-1] + 3)
ax.set_xlabel("Decade", labelpad=8)
ax.set_ylabel("Share of movies in dataset (%)", labelpad=8)
ax.set_title(
    "Comedy & Adventure Are Dominating — Smaller Genres Are Shrinking",
    pad=14, loc="left", fontsize=15
)
ax.text(0, 1.01,
        "Genre share = proportion of total output each decade. "
        "Shows which types of fantasy films studios chose to make — not raw volume.",
        transform=ax.transAxes, fontsize=10, color=GREY, style="italic")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))

fig.text(
    0.5, 0.02,
    "Takeaway: Adventure and Comedy have grown their share of the fantasy space each decade. " +"\n"+
    "Smaller genres like Horror have lost ground — suggesting studios are "
    "consolidating around blockbuster-friendly subgenres." + "\n"+
    "With the increase in anime from Japan being more mainstream, you can see a clear increase in its popularity with time",
    ha="center", va="bottom", fontsize=9.5, color=DARK, style="italic",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#F5F5F5", edgecolor="#CCCCCC")
)

out2 = f"{OUT_DIR}/charts2_genre_shift_by_decade.png"
fig.savefig(out2, dpi=180, bbox_inches="tight")
plt.close()